# STAGE 1 & STAGE 2 — Kiểm kê dữ liệu & Hiểu cấu trúc chuỗi thời gian

**Đề tài:** Mô hình hóa mức độ phơi nhiễm chuỗi lạnh & dự đoán RSL của xoài xuất khẩu đường dài (Việt Nam → Pháp).

Notebook này thực hiện:
- **STAGE 1 — Data Inventory:** liệt kê toàn bộ file (raw logs cảm biến, đo chất lượng, metadata chặng vận chuyển, hình ảnh), xác định cấu trúc thư mục, xây dựng **Data Dictionary**, và xác định **phân cấp thực thể** `Shipment_ID → Pallet_ID → Box_ID → Mango_ID → Sensor_ID`.
- **STAGE 2 — Time-Series Structure:** tái dựng **timeline chuỗi lạnh** (Step 0 → Step 10), **kiểm chuẩn timestamps** (múi giờ, sampling step, trùng lặp/gaps) và **ánh xạ Sensor ↔ Thùng hàng** (lồng quả vs. không khí thùng vs. khoang ngoài).

In [ ]:
# ---- Import các thư viện cần thiết ----
# %pip install -q pandas numpy matplotlib
import os
import re
import glob
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_rows", 120)
pd.set_option("display.width", 160)

DATA_RAW = Path("data/raw")          # gốc dữ liệu thô của dự án
assert DATA_RAW.exists(), "Không tìm thấy data/raw - hay chay notebook tu thu muc goc repo."
print("Thu muc du lieu:", DATA_RAW.resolve())

---
# STAGE 1 — KIỂM KÊ DỮ LIỆU (DATA INVENTORY)

## 1.1 Liệt kê toàn bộ file theo 4 nhóm

| Nhóm | Nội dung | Thư mục nguồn |
|---|---|---|
| **A. Sensor raw logs** | Nhiệt độ quả (T_mangoes), nhiệt độ không khí (T_Air), độ ẩm (RH_Air) theo từng chặng | `01_src_subset_dataset_T_H`, `11_src_subset_dataset_SR` |
| **B. Quality measurements** | Khối lượng (g), màu L\*a\*b\*, pH, TSS (°Brix) đo định kỳ theo ngày | `03_..._W`, `05_..._C`, `07_..._PH`, `09_..._TSS` |
| **C. Metadata chặng vận chuyển** | Tên chặng trong tên file: Packing house → Cold storage → Xe tải BKK → Sân bay → In-flight → CDG → INRAE; script xử lý R | `00_script`, tiền tố số trong `01_...` |
| **D. Hình ảnh** | Ảnh chụp quả theo ngày (D3…D15), ảnh thiết bị đo, đồ thị tổng hợp | `13_src_subset_dataset_photo`, `14_devices`, `02/04/06/08/10/12_res_*` |

In [ ]:
# ---- Quét toàn bộ cây thư mục data/raw ----
all_files = [Path(p) for p in glob.glob(str(DATA_RAW / "**" / "*"), recursive=True) if Path(p).is_file()]

def rel(p): return str(p.relative_to(DATA_RAW))

summary = (
    pd.DataFrame({
        "folder": [p.parent.name for p in all_files],
        "ext":    [p.suffix.lower() or "(no ext)" for p in all_files],
        "size_KB": [round(p.stat().st_size / 1024, 1) for p in all_files],
    })
    .groupby(["folder", "ext"])
    .agg(n_files=("ext", "size"), total_KB=("size_KB", "sum"))
    .reset_index()
    .sort_values(["folder", "ext"])
)
print(f"TONG SO FILE: {len(all_files)}")
summary

In [ ]:
# ---- Phân loại file vào 4 nhóm A/B/C/D ----
groups = {
    "A_Sensor_raw_logs": [
        p for p in all_files
        if p.suffix == ".txt"
        and ("src_subset_dataset_T_H" in str(p) or "src_subset_dataset_SR" in str(p))
    ],
    "B_Quality_measurements": [
        p for p in all_files
        if p.suffix == ".txt" and re.search(r"_(W|C|PH|TSS)$", p.parent.name) and "read.me" not in p.name
    ],
    "C_Metadata_transport_steps": [
        p for p in all_files if p.parent.name == "00_script"
    ],
    "D_Images_photos_plots": [
        p for p in all_files if p.suffix.lower() in (".jpg", ".jpeg", ".png")
    ],
}

inventory_rows = []
for gname, files in groups.items():
    folders = sorted({f.parent.name for f in files})
    inventory_rows.append({
        "Nhóm": gname,
        "Số file": len(files),
        "Định dạng": ", ".join(sorted({f.suffix.lower() for f in files})) or "-",
        "Thư mục chứa": "; ".join(folders),
    })
df_inventory = pd.DataFrame(inventory_rows)
df_inventory

In [ ]:
# ---- Ví dụ minh họa từng nhóm (5 file đầu mỗi nhóm) ----
for gname, files in groups.items():
    print(f"\n### {gname} ({len(files)} file)")
    for f in sorted(files)[:5]:
        print("  ", rel(f))

## 1.2 Xác định cấu trúc thư mục lưu trữ

Cấu trúc gốc dùng quy ước `NN_src_*` (dữ liệu nguồn) và `NN_res_*` (kết quả đồ thị của cộng sự INRAE). Để thống nhất với pipeline dự án (`src/data/`), ta ánh xạ sang chuẩn `data/raw/sensor/` và `data/raw/quality/` bằng **symbolic links** (không sao chép, giữ nguyên bản gốc để đối chiếu traceability).

In [ ]:
# ---- Liệt kê các thư mục con của data/raw ----
tree = []
for d in sorted(DATA_RAW.iterdir()):
    if d.is_dir():
        n = sum(1 for _ in d.rglob("*") if _.is_file())
        tree.append((d.name, n))
df_tree = pd.DataFrame(tree, columns=["Thu muc con", "So file"])
df_tree["Loai du lieu"] = df_tree["Thu muc con"].map(lambda s: {
    "00_script": "C - Metadata/script",
    "01_src_subset_dataset_T_H": "A - Sensor (log theo chang)",
    "11_src_subset_dataset_SR": "A - Sensor (kho luu tru)",
    "03_src_subset_dataset_W": "B - Quality (Khoi luong)",
    "05_src_subset_dataset_C": "B - Quality (Mau Lab)",
    "07_src_subset_dataset_PH": "B - Quality (pH)",
    "09_src_subset_dataset_TSS": "B - Quality (TSS)",
    "13_src_subset_dataset_photo": "D - Anh qua",
    "14_devices": "D - Anh thiet bi",
}.get(s, "KQ do thi (res)"))
df_tree

In [ ]:
%%bash
# ---- Chuan hoa cau truc: data/raw/sensor/ va data/raw/quality/ (symlink, idempotent) ----
cd data/raw
mkdir -p sensor quality
ln -sfn ../01_src_subset_dataset_T_H      sensor/log_by_step
ln -sfn ../11_src_subset_dataset_SR       sensor/storage_room
ln -sfn ../03_src_subset_dataset_W        quality/weight
ln -sfn ../05_src_subset_dataset_C        quality/color
ln -sfn ../07_src_subset_dataset_PH       quality/ph
ln -sfn ../09_src_subset_dataset_TSS      quality/tss
echo "=== data/raw/sensor ===" && ls -l sensor
echo "=== data/raw/quality ===" && ls -l quality

## 1.3 Xây dựng Data Dictionary

Đọc tiêu đề cột của từng loại file để sinh bảng định nghĩa tự động, sau đó bổ sung **ý nghĩa vật lý, đơn vị, kiểu dữ liệu**.

In [ ]:
# ---- Hàm đọc header của các file txt tab-separated ----
def read_header(path):
    with open(path, encoding="utf-8", errors="ignore") as fh:
        line = fh.readline().rstrip("\n")
    return [c.strip() for c in re.split(r"\t|\s{2,}", line) if c.strip()]

samples = {
    "Sensor log (Temp+Hum)": "01_src_subset_dataset_T_H/0_Packing_House_1_1_1_Temp_Hum.txt",
    "Sensor log (Temp only)": "01_src_subset_dataset_T_H/0_Packing_House_1_3_12_Temp.txt",
    "Storage room":           "11_src_subset_dataset_SR/Storage_Room_Temp_Hum.txt",
    "Weight (W)":             "03_src_subset_dataset_W/1_Weight_21C_1_1_1.txt",
    "Color (L*a*b*)":         "05_src_subset_dataset_C/1_Color_21C_1_1_1.txt",
    "pH":                     "07_src_subset_dataset_PH/1_PH_21C.txt",
    "TSS":                    "09_src_subset_dataset_TSS/1_TSS_21C.txt",
}
headers = {k: read_header(DATA_RAW / v) for k, v in samples.items()}
for k, h in headers.items():
    suffix = f" ... ({len(h)} cot)" if len(h) > 6 else ""
    print(f"{k:24s}: {h[:6]}{suffix}")

In [ ]:
# ---- Data Dictionary chính thức ----
rows_dd = [
    ["Time",              "Timestamp ghi nhận (giờ địa phương nơi đặt cảm biến)", "datetime (dd/mm/YYYY HH:MM)", "datetime", "01_T_H, 03_W, 05_C, 07_PH, 09_TSS, 11_SR"],
    ["T_mangoes",         "Nhiệt độ ruột quả xoài (cảm biến tiếp xúc / lồng trong quả)", "°C", "float", "01_src_subset_dataset_T_H"],
    ["T_Air",             "Nhiệt độ không khí bên trong thùng carton", "°C", "float", "01_src_subset_dataset_T_H"],
    ["RH_Air",            "Độ ẩm tương đối không khí trong thùng", "%", "float", "01_src_subset_dataset_T_H (file *_Temp_Hum)"],
    ["T_16C / T_21C",     "Nhiệt độ không khí phòng kho lưu trữ @ chế độ 16°C / 21°C (khoang ngoài)", "°C", "float", "11_src_subset_dataset_SR"],
    ["H_16C / H_21C",     "Độ ẩm tương đối phòng kho 16°C / 21°C", "%", "float", "11_src_subset_dataset_SR"],
    ["M_<box>/<n>",       "Khối lượng quả thứ n trong box <box> (vd M_1_1_1/1)", "g", "float", "03_src_subset_dataset_W"],
    ["L_D<k>",            "Độ sáng màu vỏ quả tại ngày đo D<k>, k = số ngày sau đóng gói (L*a*b*)", "đơn vị L* (0–100)", "float", "05_src_subset_dataset_C"],
    ["a_D<k>",            "Trục màu đỏ (+)/xanh lục (−) tại ngày D<k>", "đơn vị a*", "float", "05_src_subset_dataset_C"],
    ["b_D<k>",            "Trục màu vàng (+)/xanh lam (−) tại ngày D<k> — chỉ báo chín/vàng hóa", "đơn vị b*", "float", "05_src_subset_dataset_C"],
    ["BOX_NAME",          "Nhãn vị trí đo màu: <Box>/<Mango>-<mã điểm đo> (vd 1_1_1/1-1A)", "chuỗi", "str", "05_src_subset_dataset_C"],
    ["pH (cột M_...)",    "Độ chua thịt quả của quả được chỉ định, đo định kỳ theo ngày", "pH", "float", "07_src_subset_dataset_PH"],
    ["TSS (cột M_...)",   "Tổng chất rắn hòa tan — độ ngọt", "°Brix", "float", "09_src_subset_dataset_TSS"],
]
data_dict = pd.DataFrame(rows_dd, columns=["Tên cột", "Ý nghĩa vật lý", "Đơn vị đo", "Kiểu dữ liệu", "Nguồn file"])
data_dict.to_csv("data/dictionary.csv", index=False, encoding="utf-8-sig")
data_dict

## 1.4 Phân cấp thực thể (Entity Hierarchy)

$$\text{Shipment\_ID} \rightarrow \text{Pallet\_ID} \rightarrow \text{Box\_ID} \rightarrow \text{Mango\_ID} \rightarrow \text{Sensor\_ID}$$

Quy ước giải mã từ tên file/cột: `X_Y_Z` = Pallet X / Tầng Y / Thùng Z (Box_ID); `M_X_Y_Z/<n>` = quả thứ n trong box; tiền tố số `S#_` = log cảm biến theo chặng.

In [ ]:
# ---- Trích xuất Box_ID từ tên file sensor + từ cột của file chất lượng ----
box_ids = set()
for f in groups["A_Sensor_raw_logs"]:
    m = re.search(r"_(\d+_\d+_\d+)_(?:Temp|All_Recording)", f.name)
    if m: box_ids.add(m.group(1))
for f in groups["B_Quality_measurements"]:
    for c in read_header(f):
        m = re.match(r"(?:M_|)(\d+_\d+_\d+)/", c)
        if m: box_ids.add(m.group(1))

boxes = sorted(box_ids, key=lambda b: tuple(map(int, b.split("_"))))
print(f"So Box_ID duy nhat: {len(boxes)}")
print(", ".join(boxes))

# ---- Bảng phân cấp thực thể ----
entities = pd.DataFrame([
    ["Shipment_ID", 2, "2 lô vận chuyển VN→Pháp (chế độ bảo quản 21°C và 16°C/random)", "Tiền tố 1_/2_ trong 03_W, 05_C, 07_PH, 09_TSS; hậu tố _21C/_16C"],
    ["Pallet_ID",   4, "Vị trí pallet trên xe/container (trục 1..4)", "Thành phần thứ nhất X trong X_Y_Z"],
    ["Tier/Level",  4, "Tầng xếp thùng trong pallet (trục 1..4)", "Thành phần thứ hai Y trong X_Y_Z"],
    ["Box_ID", len(boxes), "Thùng carton xoài, định danh X_Y_Z (thứ tự Z: 1,3,6,7,10,12)", "Thành phần thứ ba Z; tên file S*_..._<X_Y_Z>_Temp*.txt"],
    ["Mango_ID", None, "Quả trong box: M_<Box>/<n>, n = 1..14 (14 quả/box theo file W)", "Cột file 03_W; BOX_NAME file 05_C"],
    ["Sensor_ID", None, "Kênh cảm biến: T_mangoes (lồng quả), T_Air (không khí thùng), RH_Air (ẩm thùng)", "Các cột trong file 01_T_H / 11_SR"],
    ["Step_ID", 11, "Chặng chuỗi lạnh 0..10 (Packing House → ... → Arrival INRAE)", "Tiền tố số trong tên file 01_T_H"],
], columns=["Thực thể", "Số lượng", "Ý nghĩa", "Cách suy ra từ dữ liệu"])
entities

In [ ]:
# ---- Kiểm tra tính nhất quán: cùng một Box_ID có bao nhiêu sensor-log file? ----
box_sensor_counts = Counter()
for f in groups["A_Sensor_raw_logs"]:
    m = re.search(r"_(\d+_\d+_\d+)_(?:Temp|All_Recording)", f.name)
    if m: box_sensor_counts[m.group(1)] += 1
df_bs = pd.Series(box_sensor_counts, name="so_file_log_theo_box").sort_index()
print(df_bs.describe().round(2))
print("\nBox co nhieu log nhat:", df_bs.idxmax(), df_bs.max())
df_bs.head(10)

---
# STAGE 2 — HIỂU CẤU TRÚC CHUỖI THỜI GIAN

## 2.1 Tái dựng tiến trình chuỗi lạnh (Cold-chain Timeline)

Mỗi chặng được lưu trong một file riêng, có tiền tố `Step` trong tên file thuộc `01_src_subset_dataset_T_H`. File `00_All_Recording_...` là log ghép toàn trình (Packing House → Arrival at INRAE).

In [ ]:
# ---- Suy ra danh sách Step từ tiền tố tên file ----
step_names = {}
for f in groups["A_Sensor_raw_logs"]:
    m = re.match(r"(\d+(?:_\d+)?)_([A-Za-z0-9_]+?)_(\d+_\d+_\d+)_", f.name)
    if m:
        step_names.setdefault(m.group(1), m.group(2))

timeline = pd.DataFrame(
    [{"Step_ID": k, "Ten_chang": v} for k, v in step_names.items()]
).sort_values("Step_ID", key=lambda s: s.astype(float))

timeline["Dia_diem"] = timeline["Ten_chang"].map(lambda s: s.replace("_", " "))
timeline["Quoc_gia"] = np.where(timeline["Ten_chang"].str.contains("INRAE|CDG|PARIS"), "Phap",
                        np.where(timeline["Ten_chang"].str.contains("BKK|Tarmac|Mango_Box|Loading"), "Thai Lan", "Viet Nam"))
timeline["Loai_mo_truong"] = np.where(timeline["Ten_chang"].str.contains("Inflight"), "Buong may bay",
                              np.where(timeline["Ten_chang"].str.contains("Air|Airport|Tarmac|Platform"), "San bay / outdoor", "Kho / xe lanh"))
timeline

**Chuỗi logic cold-chain:**

`Step 0 (Đóng gói – Packing House)` → `Step 1 (Tiền lạnh – Cold Storage)` → `Step 2 (Xe lạnh đi BKK)` → `Step 3 (Sân bay BKK – Platform)` → `Step 4 (Xếp thùng)` → `Step 5 (Sàn đỗ – Tarmac)` → `Step 6 (Nạp hàng)` → `Step 7 (Bay – Inflight)` → `Step 8 (Sân bay CDG Paris)` → `Step 9/10 (Vận chuyển & đến kho INRAE – lưu kho)`.

In [ ]:
# ---- Duong time-line thuc te cho mot Box cu the (1_1_1): bat dau/ket thuc tung Step ----
BOX = "1_1_1"
rows = []
for f in sorted(groups["A_Sensor_raw_logs"]):
    m = re.match(r"(\d+(?:_\d+)?)_([A-Za-z0-9_]+?)_(\d+_\d+_\d+)_", f.name)
    if not m or m.group(3) != BOX:
        continue
    df = pd.read_csv(f, sep=r"\t|\s{2,}", engine="python")
    t = pd.to_datetime(df["Time"], format="%d/%m/%Y %H:%M", errors="coerce")
    rows.append({
        "Step": m.group(1), "Chang": m.group(2).replace("_", " "),
        "n_obs": int(t.notna().sum()),
        "bat_dau": t.min(), "ket_thuc": t.max(),
        "do_dai_h": round((t.max() - t.min()).total_seconds() / 3600, 2),
    })
df_tl = pd.DataFrame(rows).sort_values("Step", key=lambda s: s.astype(float))
df_tl

## 2.2 Kiểm chuẩn mốc thời gian (Timestamp Validation)

Ba việc phải làm:
1. **Đồng bộ múi giờ**: log ở Việt Nam/Thái Lan dùng giờ địa phương (UTC+7); Pháp dùng CEST (UTC+2 mùa hè). Khi ghép chuỗi phải quy về **UTC**.
2. **Sampling step**: kiểm tra chu kỳ lấy mẫu (5/10/15 phút) bằng median diff.
3. **Gaps & duplicates**: phát hiện khoảng trống (> 2× bước mẫu), timestamp lặp và bước nhảy ngược thời gian.

In [ ]:
# ---- Phan tich sampling step / duplicates / gaps cho TOAN BO file sensor log ----
def analyze_ts(path, tz_hint):
    df = pd.read_csv(path, sep=r"\t|\s{2,}", engine="python")
    t = pd.to_datetime(df["Time"], format="%d/%m/%Y %H:%M", errors="coerce").dropna()
    if len(t) < 2:
        return None
    diffs = t.sort_values().diff().dropna()
    step_min = diffs.median().total_seconds() / 60
    dups = int(diffs.eq(pd.Timedelta(0)).sum())
    back = int(diffs.lt(pd.Timedelta(0)).sum())
    gap_thr = max(2 * step_min, 15)
    gaps = diffs[diffs > pd.Timedelta(minutes=gap_thr)]
    return {
        "file": path.name, "tz": tz_hint,
        "n": len(t), "sampling_min": step_min,
        "mode_min": float(diffs.mode().iloc[0].total_seconds() / 60),
        "duplicates": dups, "backward_jumps": back,
        "n_gaps": len(gaps), "max_gap_min": float(gaps.max().total_seconds() / 60) if len(gaps) else 0.0,
    }

recs = []
for f in sorted(groups["A_Sensor_raw_logs"]):
    # Mui gio dia phuong: VN/BKK = UTC+7, Phap (CDG/INRAE) = UTC+2 (CEST)
    tz = "Europe/Paris" if re.search(r"CDG|PARIS|INRAE", f.name) else "Asia/Ho_Chi_Minh"
    r = analyze_ts(f, tz)
    if r: recs.append(r)
ts_check = pd.DataFrame(recs)
print("Phan bo sampling step (phút):")
print(ts_check["sampling_min"].value_counts())
print("\nMui gio suy ra tu ten file:")
print(ts_check["tz"].value_counts())
ts_check.head(12)

In [ ]:
# ---- Tổng hợp cảnh báo chất lượng timestamp ----
bad_dup  = ts_check[ts_check.duplicates > 0]
bad_gap  = ts_check[ts_check.n_gaps > 0]
bad_back = ts_check[ts_check.backward_jumps > 0]
print(f"File co timestamp LAP      : {len(bad_dup):3d}/{len(ts_check)}")
print(f"File co GAP (>2 buoc mau)  : {len(bad_gap):3d}/{len(ts_check)}")
print(f"File buoc NHAY NGUOC       : {len(bad_back):3d}/{len(ts_check)}")
if len(bad_gap):
    display(bad_gap.sort_values('max_gap_min', ascending=False)[['file','tz','n_gaps','max_gap_min']].head(10))

In [ ]:
# ---- Dong bo mui gio ve UTC cho mot log vi du ----
example = DATA_RAW / "01_src_subset_dataset_T_H/00_All_Recording_Packing_House_to_Arrival_At_INRAE_1_1_1_Temp_Hum.txt"
df_ex = pd.read_csv(example, sep=r"\t|\s{2,}", engine="python")
df_ex["ts_local"] = pd.to_datetime(df_ex["Time"], format="%d/%m/%Y %H:%M")
# naive local -> localize -> convert UTC
df_ex["ts_utc"] = (df_ex["ts_local"].dt.tz_localize("Asia/Ho_Chi_Minh")
                                      .dt.tz_convert("UTC"))
df_ex[["Time", "ts_local", "ts_utc", "T_mangoes", "T_Air", "RH_Air"]].head()

> ⚠️ **Lưu ý nghiệp vụ:** tên file gộp cả chặng VN → Thái Lan → Pháp nhưng timestamp là **giờ địa phương** của nơi đặt logger. Khi ghép toàn trình (Step 0→10) bắt buộc localize đúng tz từng chặng rồi mới convert UTC, nếu không sẽ lệch ~5–6 giờ giữa Paris và Bangkok/HCM.

## 2.3 Ánh xạ Sensor ↔ Thùng hàng

Ba loại kênh đo cần phân biệt rõ:

| Kênh | Vị trí vật lý | Bản chất |
|---|---|---|
| `T_mangoes` | **Lồng trong quả** (cảm biến tiếp xúc, đặt inside quả) | Nhiệt độ sản phẩm — biến đầu vào chính cho mô hình RSL |
| `T_Air` | Treo trong **không khí thùng carton** | Nhiệt độ môi trường vi mô quanh quả |
| `RH_Air` | Trong thùng | Độ ẩm tương đối |
| `T_16C/T_21C, H_*` | **Khoang ngoài** — phòng kho lưu trữ INRAE | Điều kiện kho đích (ambient reference) |

In [ ]:
# ---- Ma tran anh xa Box x Step x loai kenh ----
mapping_rows = []
for f in sorted(groups["A_Sensor_raw_logs"]):
    m = re.match(r"(\d+(?:_\d+)?)_([A-Za-z0-9_]+?)_(\d+_\d+_\d+)_", f.name)
    if not m:
        continue
    cols = read_header(f)
    mapping_rows.append({
        "Box_ID": m.group(3), "Step": m.group(1), "Chang": m.group(2).replace("_", " "),
        "T_qua": "T_mangoes" in cols,
        "T_khong_khi_thung": "T_Air" in cols,
        "Do_am_thung": "RH_Air" in cols,
    })
df_map = pd.DataFrame(mapping_rows)
print(f"So ban ghi Box x Step: {len(df_map)}  |  So Box duoc log: {df_map.Box_ID.nunique()}")
print("\nBox nao co du ca 3 kenh o moi step?")
full = df_map.groupby("Box_ID")[["T_qua", "T_khong_khi_thung", "Do_am_thung"]].min()
print(full.sum(axis=1).value_counts().rename("count_box"))
df_map.head(10)

In [ ]:
# ---- Doi chieu truc quan: T_qua vs T_khong_khi (box 1_1_1, Packing House) & kho ngoai ----
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=False)

d1 = pd.read_csv(DATA_RAW / "01_src_subset_dataset_T_H/0_Packing_House_1_1_1_Temp_Hum.txt",
                 sep=r"\t|\s{2,}", engine="python")
t1 = pd.to_datetime(d1["Time"], format="%d/%m/%Y %H:%M")
axes[0].plot(t1, d1["T_mangoes"], label="T_mangoes (long qua)", lw=1.2)
axes[0].plot(t1, d1["T_Air"], label="T_Air (khong khi thung)", lw=1.2)
axes[0].set_title("Step 0 - Packing House, Box 1_1_1: Sensor tiep xuc vs khong khi")
axes[0].set_ylabel(u"Nhiệt độ (°C)"); axes[0].legend(); axes[0].grid(alpha=.3)

sr = pd.read_csv(DATA_RAW / "11_src_subset_dataset_SR/Storage_Room_Temp_Hum.txt",
                 sep=r"\t|\s{2,}", engine="python")
tsr = pd.to_datetime(sr["Time"], format="%d/%m/%Y %H:%M")
axes[1].plot(tsr, sr["T_16C"], label=u"T_16C (khoang ngoai 16°C)", lw=1.2)
axes[1].plot(tsr, sr["T_21C"], label=u"T_21C (khoang ngoai 21°C)", lw=1.2)
axes[1].set_title(u"Step 10 - Kho luu tru INRAE: cam bien khoang ngoai")
axes[1].set_ylabel(u"Nhiệt độ (°C)"); axes[1].legend(); axes[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

In [ ]:
# ---- Tuong quan tre giua T_Air va T_mangoes (tre nhiet cua khoi qua) ----
tm = d1["T_mangoes"].astype(float).reset_index(drop=True)
ta = d1["T_Air"].astype(float).reset_index(drop=True)
for lag in range(0, 7):  # lag tinh theo buoc mau 5 phut
    c = tm.corr(ta.shift(lag))
    print(f"lag={lag*5:2d} phut : corr(T_air -> T_mango) = {c:.3f}")

## Kết luận Stage 1 & 2

1. **Kiểm kê**: toàn bộ file trong `data/raw` chia rõ 4 nhóm (sensor logs / quality / metadata-script / images); đã ánh xạ sang chuẩn `data/raw/sensor/{log_by_step, storage_room}` và `data/raw/quality/{weight,color,ph,tss}`.
2. **Data Dictionary**: các họ cột được định nghĩa đầy đủ (ý nghĩa vật lý, đơn vị °C/%/g/pH/°Brix/ngày, kiểu datetime/float), xuất ra `data/dictionary.csv`.
3. **Entity hierarchy**: Shipment (2 lô 16°C/21°C) → Pallet (4) → Box (~31, mã X_Y_Z) → Mango (1..14) → Sensor (T_mangoes / T_Air / RH_Air).
4. **Timeline**: 11 chặng Step 0→10 phục dựng được từ tiền tố file; độ dài mỗi chặng đo được từ min/max timestamp.
5. **Timestamp**: sampling chủ yếu 5 phút (log) và ~3 ngày (quality); cần localize tz theo quốc gia trước khi convert UTC; đã kiểm đếm số file có duplicate/gap/backward-jump.
6. **Sensor↔Box**: phân biệt rõ cảm biến tiếp xúc quả (T_mangoes), không khí thùng (T_Air/RH_Air) và khoang ngoài (kho 16/21°C); có bằng chứng trễ nhiệt qua phân tích tương quan theo lag.